<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m7_02_cross_validation_choices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M7.02 — The same model, three different scores

In the lab we saw that the way the data is cut into folds must match the
way the model will be used: stratified folds for rare classes, grouped
folds when rows belong together, ordered folds when time matters. In
workbook M7.01 a model predicted the severity of Parkinson's disease
from voice recordings with an error of 2.4 points, against 8.7 for the
baseline. In this workbook you test **the same model** in two other
ways, and obtain two very different answers. All three are correct; they
answer different questions.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "harder_task": "123e36f317e5ebd8",
    "kfold_test_recordings_from": "2cd3b3b14fa41cb1",
    "model_for_new_patients": "e760bdbe0be65db9",
    "model_is_useful_for": "faa6fa70df830370",
    "n_trainings_leave_one_patient_out": "73475cb40a568e8d",
    "removing_age_and_sex_is_enough": "9390298f3fb0c5b1",
    "right_cv_for_new_patients": "002eabd5671d9c57"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Where we left off

5875 voice recordings of 42 patients; we predict `total_updrs`, the
severity of the symptoms, from 16 voice measurements, the age and the
sex. The model is a histogram gradient boosting, the baseline predicts
the mean.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_validate

recordings = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/parkinsons_telemonitoring.csv"
)
features = [
    name for name in recordings.columns if name not in ("subject", "test_time", "motor_updrs", "total_updrs")
]
data = recordings[features]
target = recordings["total_updrs"]
patient = recordings["subject"]

model = HistGradientBoostingRegressor(random_state=0)
dummy = DummyRegressor()


def error(estimator, cv, columns=features):
    """Mean absolute error over the folds of `cv`."""
    groups = None if isinstance(cv, KFold) else patient  # only grouped splitters use them
    scores = cross_validate(
        estimator, recordings[columns], target, groups=groups, cv=cv, scoring="neg_mean_absolute_error"
    )
    return -scores["test_score"].mean()


shuffled = KFold(n_splits=6, shuffle=True, random_state=0)
errors = pd.DataFrame(
    {"shuffled recordings": {"model": error(model, shuffled), "dummy": error(dummy, shuffled)}}
).T
errors.round(2)

With recordings shuffled into folds, every patient is on both sides of
every split. The model is tested on **new recordings of patients it
already knows**.

## 2. Patients the model has never seen

The hospital's plan is different: use the model for patients who join
**next year**. To imitate that, all the recordings of a patient must be
on the same side of a split. `GroupKFold` does it: we tell it which
group (patient) each row belongs to, and it never cuts a group in two.

🔮 **Predict first.** Tested on patients it has never seen, the model's
error will be **"still close to 2.4"**, **"between 2.4 and the dummy's
8.7"** or **"worse than the dummy"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["still close to 2.4", "between 2.4 and the dummy's 8.7", "worse than the dummy"])

In [ ]:
by_patient = GroupKFold(n_splits=6)
errors.loc["whole patients held out"] = {"model": error(model, by_patient), "dummy": error(dummy, by_patient)}
errors.round(2)

✏️ Which cross-validation tells the hospital what to expect for next
year's patients? Write `"KFold"` or `"GroupKFold"`. And for those
patients, is the model `"better"` or `"worse"` than always predicting
the mean?

In [ ]:
right_cv_for_new_patients = ____
model_for_new_patients = ____
check("right_cv_for_new_patients", right_cv_for_new_patients)
check("model_for_new_patients", model_for_new_patients)

The "excellent" model of workbook M7.01 is **worse than no model** for
the use the hospital had in mind. Nothing was wrong with the code, the
baseline or the permutation test. The test answered another question.

### Seeing the two ways of cutting

Each stripe below is the first 1000 recordings of the file (the first 7
patients, one after the other). Dark marks are the recordings that fall
in the test part of the first split.

In [ ]:
first_rows = 1000
fig, axes = plt.subplots(3, 1, figsize=(11, 4), sharex=True, constrained_layout=True)
axes[0].imshow(patient.iloc[:first_rows].to_numpy()[None, :], aspect="auto", cmap="tab10")
axes[0].set(yticks=[], title="which patient each recording belongs to")
for ax, (name, splitter) in zip(axes[1:], [("KFold, shuffled", shuffled), ("GroupKFold", by_patient)]):
    groups = None if isinstance(splitter, KFold) else patient
    _, test_rows = next(splitter.split(data, target, groups))
    in_test = np.isin(np.arange(first_rows), test_rows)
    ax.imshow(in_test[None, :], aspect="auto", cmap="Greys")
    ax.set(yticks=[], title=f"{name}: recordings in the test part")
_ = axes[-1].set_xlabel("recording (position in the file)")

✏️ With the shuffled `KFold`, the test recordings of one split come from
`"a few patients"` or from `"every patient"`?

In [ ]:
kfold_test_recordings_from = ____
check("kfold_test_recordings_from", kfold_test_recordings_from)

scikit-learn's example
[Visualizing cross-validation behavior](https://scikit-learn.org/1.8/auto_examples/model_selection/plot_cv_indices.html)
draws this kind of picture for every splitter.

## 3. ➕ Going further: how does the model recognise a patient?

The model was never given the patient number. But it has `age` and
`sex`:

In [ ]:
profiles = recordings.groupby("subject")[["age", "sex"]].first()
print(f"{len(profiles)} patients, {len(profiles.drop_duplicates())} different combinations of age and sex")

Age and sex together are almost a name tag. And the severity of a
patient hardly changes in six months. So a model can do very well on
shuffled recordings by learning "the 72-year-old man is at about 35",
which says nothing about any other 72-year-old man.

🔮 **Predict first.** We remove `age` and `sex` and keep only the 16
voice measurements. Will the two cross-validations now agree? `"yes"`
or `"no"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["yes", "no"])

In [ ]:
voice = [name for name in features if name not in ("age", "sex")]
voice_only = pd.DataFrame(
    {
        "shuffled recordings": {"model": error(model, shuffled, voice), "dummy": error(dummy, shuffled, voice)},
        "whole patients held out": {"model": error(model, by_patient, voice), "dummy": error(dummy, by_patient, voice)},
    }
).T
voice_only.round(2)

✏️ Is removing the columns that identify a patient enough to make the
shuffled cross-validation honest? Write `"yes"` or `"no"`.

In [ ]:
removing_age_and_sex_is_enough = ____
check("removing_age_and_sex_is_enough", removing_age_and_sex_is_enough)

A voice is a signature too: the 16 measurements of one person resemble
each other from one recording to the next. You cannot make a leaky test
honest by deleting columns one by one. The split has to respect the
groups.

## 4. ➕ Going further: a third question, about time

There is a use for which knowing the patient is **allowed**: following,
week after week, people who are already in the programme. The model
learns from each patient's first months and predicts their later
recordings.

We train on the first 70% of each patient's recording days and test on
the last 30%.

In [ ]:
from sklearn.metrics import mean_absolute_error

progress = recordings.groupby("subject")["test_time"].rank(pct=True)
early, late = progress <= 0.7, progress > 0.7
later_errors = {}
for name, estimator in [("model", model), ("dummy", dummy)]:
    estimator.fit(data[early], target[early])
    later_errors[name] = mean_absolute_error(target[late], estimator.predict(data[late]))
errors.loc["known patients, later recordings"] = later_errors
errors.round(2)

✏️ For which use is this model worth something? Write `"new patients"`
or `"following known patients"`. And which task is harder for it:
predicting recordings `"mixed among the training days"` (the shuffled
folds) or `"later than all the training days"`?

In [ ]:
model_is_useful_for = ____
harder_task = ____
check("model_is_useful_for", model_is_useful_for)
check("harder_task", harder_task)

Even for known patients, the shuffled folds were too kind: they let the
model see recordings made **after** the ones it was tested on, and
filling in between two known days is easier than predicting what comes
next. When the model is meant to predict the future, the test set must
be in the future. For a single series ordered in time, scikit-learn
offers `TimeSeriesSplit`, which always trains on the past and tests on
what follows.

One model, three scores:

| the question | how to split | error |
|---|---|---|
| fill in between recordings of known patients | shuffled `KFold` | 2.4 |
| predict later recordings of known patients | early days / later days | 4.0 |
| predict for patients never seen | `GroupKFold` | 11.6 (the dummy: 8.9) |

## 5. ➕ Going further: one fold per patient

With groups, a natural choice is to leave out **one patient at a time**:
`LeaveOneGroupOut`.

✏️ How many times is the model trained with `LeaveOneGroupOut` on this
data?

In [ ]:
n_trainings_leave_one_patient_out = ____
check("n_trainings_leave_one_patient_out", n_trainings_leave_one_patient_out)

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut

one_out = LeaveOneGroupOut()
per_patient = -cross_validate(
    model, data, target, groups=patient, cv=one_out, scoring="neg_mean_absolute_error"
)["test_score"]

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(per_patient, bins=15, edgecolor="black")
ax.set(xlabel="error for the patient left out (UPDRS points)", ylabel="number of patients")
print(f"mean {per_patient.mean():.1f}, from {per_patient.min():.1f} to {per_patient.max():.1f}")

The average hides a wide range: for some new patients the model is a few
points off, for others more than twenty. With only 42 patients, each
fold of a `GroupKFold` contains 7 of them, which is why its scores vary
so much from fold to fold. **The number of independent things in this
dataset is 42, not 5875.**

🧠 **Think.**

1. A start-up reports "R² = 0.91 for predicting Parkinson's severity
   from voice". Which question do you ask first?
2. Give an example from another field (shops, students, machines…) of a
   dataset with several rows per "individual", and say which of the
   three questions of the table a model would be asked there.
3. To improve the model for new patients, would you rather have ten
   times more recordings of these 42 patients, or 42 more patients?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the same model with shuffled and with grouped folds: the two
     numbers and what each one measures;
   - the picture of the two ways of cutting the recordings;
   - how the model recognises a patient, and why removing `age` and
     `sex` does not solve the problem;
   - the table of three questions: which split answers which.